In [4]:
import pandas as pd
import numpy as np
from functools import reduce

In [5]:
news_number = pd.read_csv("../datasets/GDELT (feral cat + Australia number).csv")
news_tone = pd.read_csv("../datasets/GDELT (feral cat + Australia tone timeline).csv")
news_volume = pd.read_csv("../datasets/GDELT (feral cat + Australia volume).csv")

In [6]:
news_number

,Date,Series,Value
0,2017-01-01,Article Count,2
1,2017-01-01,Total Monitored Articles,427906
2,2017-01-02,Article Count,0
3,2017-01-02,Total Monitored Articles,592774
4,2017-01-03,Article Count,1
...,...,...,...
6597,2026-02-02,Total Monitored Articles,132188
6598,2026-02-03,Article Count,0
6599,2026-02-03,Total Monitored Articles,121028
6600,2026-02-04,Article Count,0


In [7]:
news_tone

,Date,Series,Value
0,2017-01-01,Average Tone,-1.8331
1,2017-01-02,Average Tone,0.0000
2,2017-01-03,Average Tone,0.2915
3,2017-01-04,Average Tone,-1.1704
4,2017-01-05,Average Tone,-1.3840
...,...,...,...
3296,2026-01-31,Average Tone,0.0000
3297,2026-02-01,Average Tone,-4.1499
3298,2026-02-02,Average Tone,0.0000
3299,2026-02-03,Average Tone,0.0000


In [8]:
news_volume

,Date,Series,Value
0,2017-01-01,Volume Intensity,0.0005
1,2017-01-02,Volume Intensity,0.0000
2,2017-01-03,Volume Intensity,0.0001
3,2017-01-04,Volume Intensity,0.0024
4,2017-01-05,Volume Intensity,0.0012
...,...,...,...
3296,2026-01-31,Volume Intensity,0.0000
3297,2026-02-01,Volume Intensity,0.0027
3298,2026-02-02,Volume Intensity,0.0000
3299,2026-02-03,Volume Intensity,0.0000


In [9]:
news_total = news_number[news_number["Series"] == "Total Monitored Articles"].drop(["Series"], axis=1).rename(columns={"Value": "all articles"})
news_number = news_number[news_number["Series"] == "Article Count"].drop(["Series"], axis=1).rename(columns={"Value": "article count"})
news_total

,Date,all articles
1,2017-01-01,427906
3,2017-01-02,592774
5,2017-01-03,757916
7,2017-01-04,822862
9,2017-01-05,830028
...,...,...
6593,2026-01-31,133068
6595,2026-02-01,112692
6597,2026-02-02,132188
6599,2026-02-03,121028


In [10]:
news_tone = news_tone.drop(["Series"], axis=1).rename(columns={"Value": "average tone"})
news_volume = news_volume.drop(["Series"], axis=1).rename(columns={"Value": "volume intensity"})

In [11]:
news = [news_number, news_total, news_tone, news_volume]

news_all = reduce(lambda l, r: l.merge(r, on="Date", how="outer"), news)
news_all

,Date,article count,all articles,average tone,volume intensity
0,2017-01-01,2,427906,-1.8331,0.0005
1,2017-01-02,0,592774,0.0000,0.0000
2,2017-01-03,1,757916,0.2915,0.0001
3,2017-01-04,20,822862,-1.1704,0.0024
4,2017-01-05,10,830028,-1.3840,0.0012
...,...,...,...,...,...
3296,2026-01-31,0,133068,0.0000,0.0000
3297,2026-02-01,3,112692,-4.1499,0.0027
3298,2026-02-02,0,132188,0.0000,0.0000
3299,2026-02-03,0,121028,0.0000,0.0000


In [12]:
news_all["Date"] = pd.to_datetime(news_all["Date"])
news_all["year"] = news_all["Date"].dt.year

news_all = (
    news_all.groupby("year", as_index=False)
      .apply(lambda g: pd.Series({
          "article_count": g["article count"].sum(),
          "tone_weighted": np.average(g["average tone"], weights=g["article count"]) if g["article count"].sum() > 0 else np.nan,
          "all_articles": g["all articles"].sum(),
          "volume_intensity_mean": g["volume intensity"].mean(),
      }))
      .reset_index(drop=True)
)

news_all

C:\Users\Ксения\AppData\Local\Temp\ipykernel_20492\2733970731.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,year,article_count,tone_weighted,all_articles,volume_intensity_mean
0,2017,2014.0,-1.837383,227816569.0,0.000893
1,2018,1054.0,-1.624291,193012221.0,0.000544
2,2019,383.0,-2.575505,170537968.0,0.000232
3,2020,559.0,-2.436784,137760786.0,0.000379
4,2021,210.0,-1.724470,120343036.0,0.000171
5,2022,69.0,-2.148294,103082810.0,0.000065
6,2023,393.0,-2.858401,68084868.0,0.000656
7,2024,297.0,-0.819262,59778085.0,0.000503
8,2025,322.0,-1.579906,56019406.0,0.000650
9,2026,57.0,-5.721858,5092243.0,0.000946


In [13]:
news_all["volume_intensity"] = news_all["article_count"] / news_all["all_articles"]
news_all

,year,article_count,tone_weighted,all_articles,volume_intensity_mean,volume_intensity
0,2017,2014.0,-1.837383,227816569.0,0.000893,8.840446e-06
1,2018,1054.0,-1.624291,193012221.0,0.000544,5.460794e-06
2,2019,383.0,-2.575505,170537968.0,0.000232,2.245834e-06
3,2020,559.0,-2.436784,137760786.0,0.000379,4.057758e-06
4,2021,210.0,-1.724470,120343036.0,0.000171,1.745012e-06
5,2022,69.0,-2.148294,103082810.0,0.000065,6.693648e-07
6,2023,393.0,-2.858401,68084868.0,0.000656,5.772208e-06
7,2024,297.0,-0.819262,59778085.0,0.000503,4.968376e-06
8,2025,322.0,-1.579906,56019406.0,0.000650,5.748008e-06
9,2026,57.0,-5.721858,5092243.0,0.000946,1.119350e-05


In [15]:
news_all.to_csv("../datasets/news.csv", index=False)